In [1]:
from functools import partial
import json
import torch
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForSeq2Seq
from datasets import Dataset
from swanlab.integration.transformers import SwanLabCallback
import swanlab
from modelscope import AutoTokenizer

from peft import LoraConfig, get_peft_model, TaskType

import torch
import random
import numpy as np
import os

seed = 7
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
random.seed(seed)
np.random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)

g:\software\anaconda\envs\pytorch\Lib\site-packages\torch\cuda\__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [8]:
model_id = "qwen2.5-1.5B-Instruct"  
model_path = "G:/model_weights/models/Qwen/qwen2.5-1.5B-Instruct"
data_path = "D:/doc/my_project/lianxi/home_work/stage_3/data_set"

# 1.数据预处理

In [3]:
class TCMNERDataset():
    def __init__(self, data_path):
        self.label_set = set()
        self.data = self.load_data(data_path)
        # self.dataset = Dataset.from_list(self.data)
        self.id2label = {idx: item for idx, item in enumerate(self.label_set)}
        self.label2id = {item: idx for idx, item in self.id2label.items()}


    def load_data(self, data_path):
        dataset = []
        with open(data_path, 'r', encoding='utf-8') as f:
            trunk = f.read().split('\n\n')
            for line in trunk:
                input, output = '', []
                for idx, string in enumerate(line.split('\n')):
                    if not string.strip():  # 跳过空行
                        continue
                    text, tag = string.split(' ')  # 假设文本和标签之间用空格分隔
                    input += text
                    if tag.startswith('B-'):
                        self.label_set.add(tag[2:])
                        output.append([text, tag[2:]])  # 添加标签内容
                    elif tag.startswith('I-') and output:
                        output[-1][0] += text  # 更新标签内容
                dataset.append({'input': input, 'output': json.dumps(output, ensure_ascii=False)})  # 将标签列表转换为JSON字符串
        return dataset

In [4]:
from pathlib import Path
train_data_path = Path(data_path) / 'medical.train'
test_data_path = Path(data_path) / 'medical.test'
dev_data_path = Path(data_path) / 'medical.dev'

In [5]:
train_data = TCMNERDataset(train_data_path)
test_data = TCMNERDataset(test_data_path)
dev_data = TCMNERDataset(dev_data_path)

In [9]:
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path)#, id2label=train_data.id2label, label2id=train_data.label2id)
model.enable_input_require_grads()  # 开启梯度检查点时，要执行该方法

In [7]:
train_data.data[0], train_data.id2label, train_data.label2id

({'input': '现头昏口苦', 'output': '[["口苦", "临床表现"]]'},
 {0: 'B-方剂',
  1: 'B-中医治则',
  2: 'I-中药',
  3: 'I-中医治疗',
  4: 'I-方剂',
  5: 'I-西医诊断',
  6: 'B-中药',
  7: 'B-临床表现',
  8: 'I-中医治则',
  9: 'B-其他治疗',
  10: 'I-中医证候',
  11: 'B-西医诊断',
  12: 'I-其他治疗',
  13: 'B-中医诊断',
  14: 'B-西医治疗',
  15: 'I-西医治疗',
  16: 'I-中医诊断',
  17: 'I-临床表现',
  18: 'B-中医证候',
  19: 'B-中医治疗'},
 {'B-方剂': 0,
  'B-中医治则': 1,
  'I-中药': 2,
  'I-中医治疗': 3,
  'I-方剂': 4,
  'I-西医诊断': 5,
  'B-中药': 6,
  'B-临床表现': 7,
  'I-中医治则': 8,
  'B-其他治疗': 9,
  'I-中医证候': 10,
  'B-西医诊断': 11,
  'I-其他治疗': 12,
  'B-中医诊断': 13,
  'B-西医治疗': 14,
  'I-西医治疗': 15,
  'I-中医诊断': 16,
  'I-临床表现': 17,
  'B-中医证候': 18,
  'B-中医治疗': 19})

In [10]:
train_dataset = Dataset.from_list(train_data.data)
test_dataset = Dataset.from_list(test_data.data)
dev_dataset = Dataset.from_list(dev_data.data)

# 2.构造训练数据

In [11]:
def preprocess_function(tokenizer, examples):
    MAX_LENGTH = 512
    system_prompt = "你是一个中医药领域的命名体识别专家，你需要从给定句子中提取相关命名体，以json格式输出，例如：[['口苦', '临床表现'],['口干', '临床表现']]，其中每个元素分别表示命名体的起始位置、结束位置、文本内容和标签类型。没有实体则返回空列表[]"
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": examples['input']},
    ]
    messages = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,) # 输出会停在最后一个user，然后添加assistant的开始标记等待模型生成, 不会编码assistant的内容
    # print(messages)
    # print(examples)
    response = tokenizer(examples['output'], add_special_tokens=False)
    inputs = tokenizer(messages, truncation=True, add_special_tokens=False)

    input  = inputs['input_ids'] + response['input_ids'] + [tokenizer.pad_token_id]  # 将输入和输出拼接在一起，并添加结束标记

    attention_mask = inputs['attention_mask'] + response['attention_mask'] + [1]  # 拼接注意力掩码

    outputs = [-100] * len(inputs['input_ids']) + response['input_ids'] + [tokenizer.eos_token_id]  # 输出标签的token id，并添加结束标记

    if len(input) > MAX_LENGTH:
        input = input[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        outputs = outputs[:MAX_LENGTH]

    return {'input_ids': input, 'attention_mask': attention_mask, 'labels': outputs}

p_preprocess = partial(preprocess_function, tokenizer)

messages:
```<|im_start|>system
你是一个中医药领域的命名体识别专家，你需要从给定句子中提取相关命名体，以json格式输出，例如：[['口苦', '临床表现'],['口干', '临床表现']]，其中每个元素分别表示命名体的起始位置、结束位置、文本内容和标签类型。<|im_end|>
<|im_start|>user
现头昏口苦<|im_end|>
<|im_start|>assistant
```

In [14]:
print(train_dataset[0])
res = preprocess_function(tokenizer, train_dataset[0])
res

{'input': '现头昏口苦', 'output': '[["口苦", "临床表现"]]'}


{'input_ids': [151644,
  8948,
  198,
  56568,
  101909,
  107498,
  104799,
  109734,
  31914,
  102450,
  101057,
  3837,
  112735,
  45181,
  89012,
  22382,
  109949,
  15946,
  107439,
  78556,
  109734,
  31914,
  3837,
  23031,
  2236,
  68805,
  66017,
  3837,
  77557,
  5122,
  56330,
  39426,
  99746,
  516,
  364,
  104595,
  101107,
  84000,
  39426,
  99251,
  516,
  364,
  104595,
  101107,
  30840,
  3837,
  90919,
  103991,
  102268,
  101127,
  51463,
  109734,
  31914,
  9370,
  71618,
  26606,
  81812,
  5373,
  80565,
  81812,
  5373,
  108704,
  43815,
  33108,
  105151,
  31905,
  1773,
  80443,
  101565,
  46448,
  31526,
  34794,
  44177,
  1294,
  151645,
  198,
  151644,
  872,
  198,
  46451,
  64355,
  101616,
  39426,
  99746,
  151645,
  198,
  151644,
  77091,
  198,
  58,
  1183,
  39426,
  99746,
  497,
  330,
  104595,
  101107,
  66294,
  151643],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  

In [19]:
''.join(map(str, res['attention_mask']))

'111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111'

In [13]:
tokenizer.decode(preprocess_function(tokenizer, train_dataset[0])['input_ids'])

'<|im_start|>system\n你是一个中医药领域的命名体识别专家，你需要从给定句子中提取相关命名体，以json格式输出，例如：[[\'口苦\', \'临床表现\'],[\'口干\', \'临床表现\']]，其中每个元素分别表示命名体的起始位置、结束位置、文本内容和标签类型。没有实体则返回空列表[]<|im_end|>\n<|im_start|>user\n现头昏口苦<|im_end|>\n<|im_start|>assistant\n[["口苦", "临床表现"]]<|endoftext|>'

In [12]:
train_datasets = train_dataset.map(p_preprocess, remove_columns=train_dataset.column_names, num_proc=4)
# test_datasets = test_dataset.map(p_preprocess, remove_columns=test_dataset.column_names, num_proc=4)
dev_datasets = dev_dataset.map(p_preprocess, remove_columns=dev_dataset.column_names, num_proc=4)

Map (num_proc=4):   0%|          | 0/5259 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/657 [00:00<?, ? examples/s]

#  3.训练

In [13]:
fine_tuning_config = LoraConfig(task_type=TaskType.CAUSAL_LM, 
                                target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
                                r=8, 
                                lora_alpha=16, 
                                inference_mode=False, 
                                lora_dropout=0.1)

peft_model = get_peft_model(model, fine_tuning_config)
peft_model.print_trainable_parameters()

trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945


In [14]:
peft_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
  

In [15]:
args = TrainingArguments(
    output_dir="G:/model_weights/models/checkpoint/qwen2.5-1.5-tcm-ner2",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4, # 积累多少batch后更新一次权重，主要用于解决显存不足的问题
    eval_strategy="epoch",
    logging_steps=10,
    num_train_epochs=2,
    save_steps=100,
    learning_rate=1e-4,
    save_on_each_node=True,
    gradient_checkpointing=True,
    report_to="none",
)

In [ ]:
n1lKeqMVw5Ur7zirkQJlq

In [16]:
swanlab_callback = SwanLabCallback(
    project="Qwen2.5-NER-fintune",
    experiment_name="Qwen2-1.5B-Instruct",
    description="使用通义千问Qwen2-1.5B-Instruct模型在中医药NER数据集上微调，实现关键实体识别任务。",
    config={
        "model": model_id,
        "model_dir": model_path,
        "dataset": "qgyd2021/chinese_ner_sft",
    },
)

In [17]:
trainer = Trainer(
    model=peft_model,
    args=args,
    train_dataset=train_datasets,
    eval_dataset=dev_datasets,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
    callbacks=[swanlab_callback],
)


In [18]:
trainer.train()

Output()

Output()

swanlab: Tracking run with swanlab version 0.7.8

swanlab: Run data will be saved locally in 
d:\doc\my_project\lianxi\home_work\stage_3\swanlog\run-20260222_102538-yjrk3ssohwo4mkcse3ajf

swanlab: 👋 Hi yuerg1230,welcome to swanlab!

swanlab: Syncing run Qwen2-1.5B-Instruct to the cloud

swanlab: 🏠 View project at https://swanlab.cn/@yuerg1230/Qwen2.5-NER-fintune

swanlab: 🚀 View run at https://swanlab.cn/@yuerg1230/Qwen2.5-NER-fintune/runs/yjrk3ssohwo4mkcse3ajf

Epoch,Training Loss,Validation Loss
1,0.126604,0.122962
2,0.095028,0.110986


TrainOutput(global_step=658, training_loss=0.1349273352093972, metrics={'train_runtime': 801.0203, 'train_samples_per_second': 13.131, 'train_steps_per_second': 0.821, 'total_flos': 1.467532752873984e+16, 'train_loss': 0.1349273352093972, 'epoch': 2.0})

样本数量是5259，训练的时候进度提示总共是658，原因？\
`per_device_train_batch_size` = 4 每次训练4个样本 \
`gradient_accumulation_steps` = 4 每4个batch更新一次权重 \
`num_train_epochs`=2              总共训练两轮 \
训练器显示的步数是权重更新的次数\
$5259 \div 4 \div 4 \times 2 \approx 658$

In [21]:
swanlab.finish()

swanlab: 🏠 View project at https://swanlab.cn/@yuerg1230/Qwen2.5-NER-fintune

swanlab: 🚀 View run at https://swanlab.cn/@yuerg1230/Qwen2.5-NER-fintune/runs/ily2vcbjlf3ixxfg333l2

# 4.加载检查点Lora

In [21]:
from peft import  PeftModel
base_model = AutoModelForCausalLM.from_pretrained(model_path).cuda()
tokenizer = AutoTokenizer.from_pretrained(model_path)
peft_model = PeftModel.from_pretrained(base_model, "G:/model_weights/models/checkpoint/qwen2.5-1.5-tcm-ner/checkpoint-658").cuda()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [22]:
from pathlib import Path
test_data_path = Path(data_path) / 'medical.test'

test_data = TCMNERDataset(test_data_path)

In [23]:
def f1_score(set_a, set_b):
    """ 严格的F1 分数计算，只有当索引位置、实体、标签预测完全正确时才算TP，否则算FP和FN """
    TP = len(set_a & set_b)
    FP = len(set_b - set_a)
    FN = len(set_a - set_b)
    
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return f1


## 批量样本推断

In [32]:
set_a_strict = set()
set_b_strict = set()
set_a_relax = set()
set_b_relax = set()


tag_dict = {}
for entity in test_data.label_set:
    tag_dict[entity] = [set(), set()]  # [label, label_predicted]

def predict_collect(predictions, references):
    """ 预测结果收集函数，返回严格匹配和宽松匹配的实体集合 """
    set_a_strict = set()  # 严格匹配：索引位置、实体内容、标签类型都必须完全匹配
    set_b_strict = set()  # 模型预测的严格匹配结果
    
    for item in references:
        item = json.loads(item)
        for label in item:
            print(label)
            tag_dict[label[1]][0].add(tuple(label))  # 仅比较实体内容和标签类型，忽略位置
            set_a_strict.add(tuple(label))

    for output in predictions:
        try:
            output = json.loads(output.strip().replace('\n', '').replace('“', '"').replace('”', '"'))  # 去除多余的换行符和空格

            for item in output:
                if item[1] in tag_dict:
                    tag_dict[item[1]][1].add(tuple(item))
                set_b_strict.add(tuple(item))
        except:
            import traceback
            print(references)
            print('--1')
            print(output)
            print('--2')
            traceback.print_exc()
    
    return set_a_strict, set_b_strict,

In [25]:
tag_dict

{'中药': [set(), set()],
 '西医诊断': [set(), set()],
 '中医证候': [set(), set()],
 '方剂': [set(), set()],
 '中医诊断': [set(), set()],
 '临床表现': [set(), set()],
 '中医治则': [set(), set()],
 '中医治疗': [set(), set()],
 '西医治疗': [set(), set()],
 '其他治疗': [set(), set()]}

In [33]:
import tqdm

from torch.utils.data import DataLoader

test_dataset = DataLoader(test_data.data, batch_size=32, shuffle=False)

system_prompt = '你是一个中医药领域的命名体识别专家，你需要从给定句子中提取相关命名体，以json格式输出，例如：[["口苦", "临床表现"],["口干", "临床表现"]]，其中每个元素分别表示实体和标签类型。如果检测不到实体则返回空列表[]'

pbar = tqdm.tqdm(total=len(test_dataset), desc="Evaluating on test set")

for batch_sample in test_dataset:
    inputs = []
    for i in batch_sample['input']:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": i},
        ]
        input = tokenizer.apply_chat_template(messages, padding_side='left', tokenize=False, add_generation_prompt=True) # 输出会停在最后一个user，然后添加assistant的开始标记等待模型生成, 不会编码assistant的内容
        inputs.append(input)
    inputs = tokenizer(inputs, padding_side='left', padding=True, truncation=True, return_tensors="pt").to(peft_model.device)

    with torch.no_grad():
        outputs = peft_model.generate(inputs.input_ids, max_new_tokens=512)
    
    output = outputs[:, inputs.input_ids.shape[1]:]  # 获取生成的部分
    decoded_outputs = tokenizer.batch_decode(output, skip_special_tokens=True)

    set_a_strict, set_b_strict = predict_collect(decoded_outputs, batch_sample['output'])

Evaluating on test set:   0%|          | 0/21 [08:44<?, ?it/s]


['黄疸', '中医诊断']
['法莫替丁', '西医治疗']
['胃食管反流病', '西医诊断']
['疏肝行气', '中医治则']
['调神解郁', '中医治则']
['推拿', '中医治疗']
['腹泻', '临床表现']
['腹痛', '临床表现']
['功能性便秘', '西医诊断']
['头晕', '临床表现']
['腹胀胁满', '临床表现']
['大便溏薄', '临床表现']
['舌苔薄黄腻', '临床表现']
['脉细微弦', '临床表现']
['七方胃痛胶囊', '中医治疗']
['肝郁脾虚', '中医证候']
['腹泻', '临床表现']
['枳术丸加味', '方剂']
['功能性便秘', '西医诊断']
['大便', '临床表现']
['舌苔黄腻', '临床表现']
['瘀毒根深', '中医证候']
['疏香灸', '中医治疗']
['便秘型肠易激综合征', '西医诊断']
['当归', '中药']
['痛泻要方加味', '方剂']
['心理疗法', '其他治疗']
['金银花', '中药']
['连翘', '中药']
['牛蒡子', '中药']
['荆芥', '中药']
['防风', '中药']
['藿香', '中药']
['佩兰', '中药']
['碧玉散', '中药']
['桔梗', '中药']
['枳壳', '中药']
['身面发黄', '临床表现']
['目黄如金', '临床表现']
['脘痞纳差', '临床表现']
['厌食泛恶', '临床表现']
['口苦', '临床表现']
['尿如柏汁', '临床表现']
['精神萎顿', '临床表现']
['胸胁疼痛', '临床表现']
['苔黄厚', '临床表现']
['腹泻型肠易激综合征', '西医诊断']
['头晕', '临床表现']
['穴位埋线', '中医治疗']
['肝郁气滞', '中医证候']
['血活素', '中医治疗']
['非酒精性脂肪肝', '西医诊断']
['急黄', '中医诊断']
['毒陷心包', '中医证候']
['香砂枳术颗粒', '中医治疗']
['脾胃虚弱', '中医证候']
['针刺', '中医治疗']
['功能性消化不良', '西医诊断']
['针刺', '中医治疗']
['针刺', '中医治疗']
['功能性消化不良', '西医诊断']
['子午流注

Traceback (most recent call last):
  File "C:\Users\lkjx0\AppData\Local\Temp\ipykernel_31488\2331773926.py", line 25, in predict_collect
    output = json.loads(output.strip().replace('\n', '').replace('“', '"').replace('”', '"'))  # 去除多余的换行符和空格
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "g:\software\anaconda\envs\pytorch\Lib\json\__init__.py", line 346, in loads
    return _default_decoder.decode(s)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "g:\software\anaconda\envs\pytorch\Lib\json\decoder.py", line 338, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "g:\software\anaconda\envs\pytorch\Lib\json\decoder.py", line 354, in raw_decode
    obj, end = self.scan_once(s, idx)
               ^^^^^^^^^^^^^^^^^^^^^^
json.decoder.JSONDecodeError: Expecting ',' delimiter: line 1 column 98 (char 97)
Traceback (most recent call last):
  File "C:\Users\lkjx0\AppD

['和胃冲剂', '方剂']
['肝胃不和', '中医证候']
['功能性消化不良', '西医诊断']
['腹胀', '临床表现']
['尿少', '临床表现']
['恶心呕吐', '临床表现']
['苔黄', '临床表现']
['肝内小结石', '西医诊断']
['肝硬化腹水', '西医诊断']
['积聚', '中医诊断']
['臌胀', '中医诊断']
['半夏厚朴汤加味', '方剂']
['功能性消化不良', '西医诊断']
['超声离子导入', '其他治疗']
['腹痛', '临床表现']
['腹胀', '临床表现']
['脾胃虚弱', '中医证候']
['功能性消化不良', '西医诊断']
['香砂六君子汤', '方剂']
['贴敷', '中医治疗']
['香砂六君子汤', '方剂']
['针刺', '中医治疗']
['针刺', '中医治疗']
['慢性功能性便秘', '西医诊断']
['儿童功能性消化不良', '西医诊断']
['滋阴润肠口服液', '中医治疗']
['老年功能性便', '西医诊断']
['脾胃气虚', '中医证候']
['功能性消化不良', '西医诊断']
['梅花灸', '中医治疗']
['穴位贴敷', '中医治疗']
['功能性消化不良', '西医诊断']
['痰瘀互结', '中医证候']
['非酒精性脂肪肝', '西医诊断']
['退黄', '中医治则']
['浮肿', '临床表现']
['中风后遗症功能性便秘', '西医诊断']
['清肝和胃饮', '方剂']
['马来酸曲美布汀片', '西医治疗']
['肝胃郁热', '中医证候']
['功能性消化不良', '西医诊断']
['健胃消胀颗粒', '中医治疗']
['功能性消化不良', '西医诊断']
['丝瓜络', '中药']
['增液承气汤加减', '方剂']
['喘脱', '中医诊断']
['水鼓', '中医诊断']
['黄疸', '中医诊断']
['脾胃湿热', '中医证候']
['慢性萎缩性胃炎', '西医诊断']
['肠易激综合征', '西医诊断']
['非酒精性脂肪肝', '西医诊断']
['水臌', '中医诊断']
['益肠汤', '方剂']
['柴胡', '中药']
['党参', '中药']
['白术', '中药']
['陈皮', '中药']
['茯苓', '中

Traceback (most recent call last):
  File "C:\Users\lkjx0\AppData\Local\Temp\ipykernel_31488\2331773926.py", line 28, in predict_collect
    if item[1] in tag_dict:
       ~~~~^^^
IndexError: list index out of range
Traceback (most recent call last):
  File "C:\Users\lkjx0\AppData\Local\Temp\ipykernel_31488\2331773926.py", line 25, in predict_collect
    output = json.loads(output.strip().replace('\n', '').replace('“', '"').replace('”', '"'))  # 去除多余的换行符和空格
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "g:\software\anaconda\envs\pytorch\Lib\json\__init__.py", line 346, in loads
    return _default_decoder.decode(s)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "g:\software\anaconda\envs\pytorch\Lib\json\decoder.py", line 338, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "g:\software\anaconda\envs\pytorch\Lib\json\decoder.py", line 356, in raw_decode
    

...

Traceback (most recent call last):
  File "C:\Users\lkjx0\AppData\Local\Temp\ipykernel_31488\2331773926.py", line 25, in predict_collect
    output = json.loads(output.strip().replace('\n', '').replace('“', '"').replace('”', '"'))  # 去除多余的换行符和空格
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "g:\software\anaconda\envs\pytorch\Lib\json\__init__.py", line 346, in loads
    return _default_decoder.decode(s)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "g:\software\anaconda\envs\pytorch\Lib\json\decoder.py", line 338, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "g:\software\anaconda\envs\pytorch\Lib\json\decoder.py", line 354, in raw_decode
    obj, end = self.scan_once(s, idx)
               ^^^^^^^^^^^^^^^^^^^^^^
json.decoder.JSONDecodeError: Expecting ',' delimiter: line 1 column 8 (char 7)
Traceback (most recent call last):
  File "C:\Users\lkjx0\AppDat

['针刺', '中医治疗']
['耳压', '中医治疗']
['面色苍白', '临床表现']
['少气懒言', '临床表现']
['贫血貌', '临床表现']
['面色少泽', '临床表现']
['白睛轻度黄染', '临床表现']
['苔白厚腻', '临床表现']
['耳穴贴压', '中医治疗']
['心悸', '中医诊断']
['瘀阻心脉', '中医证候']
['麻仁软胶囊', '中医治疗']
['热积证功能性便秘', '西医诊断']
['茵陈', '中药']
['黄芩', '中药']
['枳壳', '中药']
['姜黄', '中药']
['大黄', '中药']
['茯苓', '中药']
['茅根', '中药']
['椒目', '中药']
['银花', '中药']
['芦根', '中药']
['生地', '中药']
['四逆散', '方剂']
['肝胃郁热', '中医证候']
['食疗', '其他治疗']
['吗丁啉', '西医治疗']
['小儿功能性消化不良', '西医诊断']
['吗丁啉', '西医治疗']
['指针', '中医治疗']
['竹药罐', '中医治疗']
['脾胃虚弱', '中医证候']
['功能性消化不良', '西医诊断']
['发热', '临床表现']
['畏寒', '临床表现']
['寒战', '临床表现']
['消瘦', '临床表现']
['苔薄黄', '临床表现']
['枸杞', '中药']
['脾阳虚', '中医证候']
['大腹胀', '临床表现']
['齿衄', '临床表现']
['郁金', '中药']
['枳壳', '中药']
['川木香', '中药']
['功能性消化不良', '西医诊断']
['利湿', '中医治则']
['清热解毒', '中医治则']
['退黄', '中医治则']
['耳穴贴压', '中医治疗']
['功能性消化不良', '西医诊断']
['功能性消化不良', '西医诊断']
['功能性消化不良', '西医诊断']
['气阴两虚', '中医证候']
['芪榔方', '方剂']
['温针灸', '中医治疗']
['慢性萎缩性胃炎', '西医诊断']
['脘腹胀满', '临床表现']
['胃脘隐痛', '临床表现']
['纳差食少', '临床表现']
['嗳气呃逆', '临床表现']
['功能性消化不良', '

Traceback (most recent call last):
  File "C:\Users\lkjx0\AppData\Local\Temp\ipykernel_31488\2331773926.py", line 25, in predict_collect
    output = json.loads(output.strip().replace('\n', '').replace('“', '"').replace('”', '"'))  # 去除多余的换行符和空格
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "g:\software\anaconda\envs\pytorch\Lib\json\__init__.py", line 346, in loads
    return _default_decoder.decode(s)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "g:\software\anaconda\envs\pytorch\Lib\json\decoder.py", line 338, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "g:\software\anaconda\envs\pytorch\Lib\json\decoder.py", line 356, in raw_decode
    raise JSONDecodeError("Expecting value", s, err.value) from None
json.decoder.JSONDecodeError: Expecting value: line 1 column 3 (char 2)
Traceback (most recent call last):
  File "C:\Users\lkjx0\AppData\Local\Temp\ip

['慢性萎缩性胃炎', '西医诊断']
['气阴不足', '中医证候']
['麻枳降浊方', '方剂']
['心悸', '临床表现']
['乏力', '临床表现']
['肝硬化腹水', '西医诊断']
['功能性消化不良', '西医诊断']
['四磨汤', '方剂']
['肝脾不和', '中医证候']
['香砂六君子汤加味', '方剂']
['木香', '中药']
['九香虫', '中药']
['砂仁', '中药']
['党参', '中药']
['炒白术', '中药']
['姜半夏', '中药']
['陈皮', '中药']
['脾胃虚寒', '中医证候']
['功能性消化不良', '西医诊断']
['中虚外寒侵袭', '中医证候']
['柴胡疏肝散', '方剂']
['萎缩性胃炎', '西医诊断']
['脾胃虚弱', '中医证候']
['双歧三联活菌', '西医治疗']
['成人功能性便秘', '西医诊断']
['腹水', '西医诊断']
['善胃Ⅰ号方', '方剂']
['血瘀热毒', '中医证候']
['胃癌前病变', '西医诊断']
['脾胃虚寒', '中医证候']
['肝郁气滞', '中医证候']
['脾胃湿热', '中医证候']
['寒热错杂', '中医证候']
['寒热错杂', '中医证候']
['慢性萎缩性胃炎', '西医诊断']
['半夏泻心汤加味', '方剂']
['柴平舒胃汤加减', '方剂']
['功能性消化不良', '西医诊断']
['柴胡疏肝散', '方剂']
['功能性消化不良', '西医诊断']
['心肺阴性', '临床表现']
['腹胀', '临床表现']
['舌质红', '临床表现']
['苔薄黄', '临床表现']
['脉弦数有力', '临床表现']
['[["慢性萎缩性胃炎", "西医诊断"], ["气阴不足", "中医证候"]]', '[["麻枳降浊方", "方剂"]]', '[["心悸", "临床表现"]]', '[["乏力", "临床表现"], ["肝硬化腹水", "西医诊断"]]', '[["功能性消化不良", "西医诊断"], ["四磨汤", "方剂"], ["肝脾不和", "中医证候"]]', '[["香砂六君子汤加味", "方剂"], ["木香", "中药"], ["九香虫", "中药"], ["砂仁", "中药"

Traceback (most recent call last):
  File "C:\Users\lkjx0\AppData\Local\Temp\ipykernel_31488\2331773926.py", line 25, in predict_collect
    output = json.loads(output.strip().replace('\n', '').replace('“', '"').replace('”', '"'))  # 去除多余的换行符和空格
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "g:\software\anaconda\envs\pytorch\Lib\json\__init__.py", line 346, in loads
    return _default_decoder.decode(s)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "g:\software\anaconda\envs\pytorch\Lib\json\decoder.py", line 338, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "g:\software\anaconda\envs\pytorch\Lib\json\decoder.py", line 354, in raw_decode
    obj, end = self.scan_once(s, idx)
               ^^^^^^^^^^^^^^^^^^^^^^
json.decoder.JSONDecodeError: Expecting ',' delimiter: line 1 column 11 (char 10)
Traceback (most recent call last):
  File "C:\Users\lkjx0\AppD

In [30]:
len(set_a_strict),  len(set_b_strict)

(38, 35)

In [27]:
f1_score(set_a_strict, set_b_strict), f1_score(set_a_relax, set_b_relax)

(0.45525902668759816, 0)

## 单样本推断

In [ ]:
import tqdm
system_prompt = "你是一个中医药领域的命名体识别专家，你需要从给定句子中提取相关命名体，以json格式输出，例如：[[', 4, '口苦', '临床表现'],[5, 6, '口干', '临床表现']]，其中每个元素分别表示命名体的起始位置、结束位置、文本内容和标签类型。如果没有则返回空列表[]"

set_a_strict = set()
set_b_strict = set()
set_a_relax = set()
set_b_relax = set()

pbar = tqdm.tqdm(total=len(test_data.data), desc="Evaluating on test set")

for i in test_data.data:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"{i['input']}"},
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) # 输出会停在最后一个user，然后添加assistant的开始标记等待模型生成, 不会编码assistant的内容
    inputs = tokenizer([inputs], return_tensors="pt").to(peft_model.device)
    with torch.no_grad():
        outputs = peft_model.generate(inputs.input_ids, max_new_tokens=512)
    
    output = outputs[0][len(inputs.input_ids[0]):]  # 获取生成的部分
    decoded_output = tokenizer.decode(output, skip_special_tokens=True)
    decoded_output = json.loads(decoded_output.strip().replace('\n', ''))  # 去除多余的换行符和空格
    print(decoded_output)
    print(i['output'])


    for item in json.loads(i['output']):
        set_a_strict.add(tuple(item))
        set_a_relax.add((item[2], item[3]))  # 仅比较实体内容和标签类型，忽略位置

    try:
        for item in decoded_output:
            set_b_strict.add(tuple(item))
            set_b_relax.add((item[2], item[3]))  # 仅比较实体内容和标签类型，忽略位置
    except:
        import traceback
        print(i['output'])
        print(decoded_output)
        traceback.print_exc()
    finally:
        pbar.update(1)


In [78]:
del peft_model

In [79]:
import gc
gc.collect()

51734

In [80]:
torch.cuda.empty_cache()